# LeetCode 1832 — Check if the Sentence Is Pangram bằng Python

Link đề: https://leetcode.com/problems/check-if-the-sentence-is-pangram/

Độ khó: **Easy** · Chủ đề: Hash Table, String

Skeleton chữ ký LeetCode:

```python
class Solution:
    def checkIfPangram(self, sentence: str) -> bool:

```

**Mục tiêu học**

1. Hiểu khái niệm **pangram** và cách chuyển nó thành câu hỏi "đủ 26 chữ cái chưa?".
2. Làm quen `str`, toán tử `in`, `string.ascii_lowercase`, `set`, `ord()`/`chr()`.
3. Học 5 cách giải tăng dần: duyệt 26 chữ với `in` → `set` → mảng đánh dấu 26 ô → **bitmask** → dừng sớm.
4. Hiểu toán tử bit `<<`, `|`, `&` qua ví dụ nhỏ có in nhị phân.
5. Trace, chứng minh, phân tích Big-O, và luyện 6 bài tập chuyển giao kỹ thuật "đánh dấu đã thấy".

## 1. Bài toán yêu cầu gì?

**Pangram** là câu chứa **mỗi chữ cái tiếng Anh (a…z) ít nhất một lần**.

Cho chuỗi `sentence` chỉ gồm chữ thường tiếng Anh, trả `True` nếu là pangram, ngược lại `False`.

| `sentence` | Kết quả | Vì sao |
|---|:---:|---|
| `"thequickbrownfoxjumpsoverthelazydog"` | `True` | có đủ 26 chữ |
| `"leetcode"` | `False` | chỉ có l, e, t, c, o, d |

**Constraints:** `1 <= sentence.length <= 1000`, chỉ chữ thường.

Hệ quả rất nhanh: nếu `len(sentence) < 26` thì chắc chắn **không** thể đủ 26 chữ khác nhau.

In [1]:
s1 = "thequickbrownfoxjumpsoverthelazydog"
s2 = "leetcode"
print(len(s1), len(s2))
print("leetcode dài < 26 nên chắc chắn False:", len(s2) < 26)

35 8
leetcode dài < 26 nên chắc chắn False: True


## 2. Nền tảng Python

### 2.1. Chuỗi, duyệt ký tự và toán tử `in`

`for ch in s` lấy lần lượt từng ký tự. `"a" in s` trả `True` nếu ký tự (hoặc chuỗi con) xuất hiện trong `s` — bên trong Python **duyệt** chuỗi nên tốn `O(len(s))`.

In [2]:
s = "leetcode"
for ch in s[:4]:
    print(ch, end=" ")
print()
print("'e' in s ->", "e" in s)
print("'z' in s ->", "z" in s)
print("'code' in s ->", "code" in s)   # in cũng kiểm tra chuỗi con

l e e t 
'e' in s -> True
'z' in s -> False
'code' in s -> True


### 2.2. `string.ascii_lowercase`

Module chuẩn `string` có sẵn hằng `ascii_lowercase = "abcdefghijklmnopqrstuvwxyz"`. Dùng nó tránh gõ tay (và gõ sót) 26 chữ.

In [3]:
import string
print(string.ascii_lowercase)
print("số chữ:", len(string.ascii_lowercase))

abcdefghijklmnopqrstuvwxyz
số chữ: 26


### 2.3. `set` — tập hợp không trùng lặp

`set(s)` tạo tập hợp các ký tự **khác nhau** của `s`. Phần tử lặp lại chỉ giữ một lần. Kiểm tra `x in some_set` nhanh **trung bình O(1)** nhờ băm (hash).

In [4]:
letters = set("leetcode")
print(letters)                 # thứ tự in có thể khác nhau giữa các lần chạy
print("số chữ khác nhau:", len(letters))
print("'t' in letters:", "t" in letters)
letters.add("z")
print("sau add('z'):", sorted(letters))

{'l', 't', 'e', 'd', 'o', 'c'}
số chữ khác nhau: 6
't' in letters: True
sau add('z'): ['c', 'd', 'e', 'l', 'o', 't', 'z']


`sorted(tập)` trả về list đã sắp xếp — tiện để in ra theo thứ tự ổn định.

### 2.4. `ord()` và `chr()` — chữ cái ↔ số

`ord("a") = 97`, `ord("b") = 98`, … Vậy `ord(ch) - ord("a")` cho **vị trí 0..25** của chữ cái. `chr` làm ngược lại.

In [5]:
for ch in "abcz":
    print(ch, "ord =", ord(ch), "-> vị trí", ord(ch) - ord("a"))
print("chr(97 + 2) =", chr(97 + 2))

a ord = 97 -> vị trí 0
b ord = 98 -> vị trí 1
c ord = 99 -> vị trí 2
z ord = 122 -> vị trí 25
chr(97 + 2) = c


### 2.5. List đánh dấu `[False] * 26`

`[False] * 26` tạo list 26 ô đều `False`. Ô `i` sẽ trả lời câu hỏi "đã thấy chữ thứ `i` chưa?". `all(list)` trả `True` nếu mọi ô đều `True`.

In [6]:
seen = [False] * 26
seen[ord("c") - ord("a")] = True
print(seen[:5])
print("all(seen) =", all(seen))
print("sum(seen) =", sum(seen), "(True được tính là 1)")

[False, False, True, False, False]
all(seen) = False
sum(seen) = 1 (True được tính là 1)


## 3. Cách 1 — duyệt 26 chữ cái, hỏi `in`

Cách tự nhiên nhất: "với mỗi chữ từ a tới z, nó có trong câu không? Thiếu một chữ là trả `False`."

In [7]:
def pangram_check_each_letter(sentence: str) -> bool:
    for ch in string.ascii_lowercase:
        if ch not in sentence:
            return False
    return True

print(pangram_check_each_letter(s1), pangram_check_each_letter(s2))

True False


### 3.1. Viết gọn bằng `all(...)`

`all(ch in sentence for ch in string.ascii_lowercase)` là **generator expression**: tạo từng giá trị `True/False` theo nhu cầu, và `all` **dừng sớm** khi gặp `False` đầu tiên.

In [8]:
def pangram_all(sentence: str) -> bool:
    return all(ch in sentence for ch in string.ascii_lowercase)

print(pangram_all(s1), pangram_all(s2))
# Minh họa dừng sớm: đếm số lần hỏi
count = 0
def asked(ch):
    global count
    count += 1
    return ch in s2
print(all(asked(ch) for ch in string.ascii_lowercase), "| số lần hỏi:", count)

True False
False | số lần hỏi: 1


Với `"leetcode"`, `all` chỉ hỏi `a` là đã gặp `False` nên dừng ngay sau 1 lần.

**Chi phí:** 26 lần × mỗi lần `in` duyệt tới `n` ký tự ⇒ `O(26·n)` = `O(n)` vì 26 là hằng số. Nhưng câu lại bị duyệt **nhiều lần**. Ta muốn đọc câu **đúng một lần**.

### 3.2. Nút thắt và insight

Nút thắt: cách 1 lặp lại việc quét câu cho từng chữ cái.

Insight: **đảo chiều câu hỏi**. Thay vì "chữ này có trong câu không?", hãy đọc câu **một lần** và **ghi lại** mọi chữ đã thấy. Cuối cùng chỉ cần hỏi: "đã thấy đủ 26 chữ khác nhau chưa?"

Đây là pattern **"đánh dấu đã thấy" (seen / visited)** — dùng `set`, list boolean, hoặc bitmask.

In [9]:
seen = set()
for ch in "leetcode":
    seen.add(ch)
    print(f"đọc {ch!r}: seen có {len(seen)} chữ -> {sorted(seen)}")

đọc 'l': seen có 1 chữ -> ['l']
đọc 'e': seen có 2 chữ -> ['e', 'l']
đọc 'e': seen có 2 chữ -> ['e', 'l']
đọc 't': seen có 3 chữ -> ['e', 'l', 't']
đọc 'c': seen có 4 chữ -> ['c', 'e', 'l', 't']
đọc 'o': seen có 5 chữ -> ['c', 'e', 'l', 'o', 't']
đọc 'd': seen có 6 chữ -> ['c', 'd', 'e', 'l', 'o', 't']
đọc 'e': seen có 6 chữ -> ['c', 'd', 'e', 'l', 'o', 't']


`{ch!r}` trong f-string in **repr** của giá trị — với chuỗi sẽ có dấu nháy, giúp nhìn rõ ký tự.

## 4. Cách 2 — `set` (lời giải chính)

Đoạn 1: tạo tập `seen = set(sentence)` (Python tự duyệt chuỗi một lần).

Đoạn 2: so sánh `len(seen) == 26`. Vì đề **đảm bảo** chỉ có chữ thường, mọi phần tử của `seen` đều là chữ cái, nên đủ 26 phần tử nghĩa là đủ a…z.

In [10]:
def pangram_set(sentence: str) -> bool:
    return len(set(sentence)) == 26

for s in [s1, s2, string.ascii_lowercase, "a" * 1000]:
    print(repr(s[:20]), "->", pangram_set(s))

'thequickbrownfoxjump' -> True
'leetcode' -> False
'abcdefghijklmnopqrst' -> True
'aaaaaaaaaaaaaaaaaaaa' -> False


### 4.1. Biến thể an toàn hơn: phép tập con `<=`

Nếu input **có thể** chứa ký tự khác (dấu cách, chữ hoa, số), đếm `len == 26` sẽ sai. Khi đó dùng phép **tập con**: `set(alphabet) <= set(sentence)` nghĩa là "mọi chữ cái đều nằm trong câu".

In [11]:
alphabet = set(string.ascii_lowercase)
messy = "The quick brown fox jumps over the lazy dog!"
print("len(set(messy)) =", len(set(messy)), "(chữ hoa, dấu cách, ! cũng bị đếm)")
print("đếm 26:", len(set(messy)) == 26)
print("tập con (sau lower):", alphabet <= set(messy.lower()))
print("{'a','b'} <= {'a','b','c'}:", {"a", "b"} <= {"a", "b", "c"})

len(set(messy)) = 29 (chữ hoa, dấu cách, ! cũng bị đếm)
đếm 26: False
tập con (sau lower): True
{'a','b'} <= {'a','b','c'}: True


## 5. Cách 3 — mảng đánh dấu 26 ô (theo gợi ý đề)

Thay `set` bằng list 26 ô, chỉ số tính bằng `ord(ch) - ord("a")`. Thêm biến `count` đếm số chữ **mới** để có thể **dừng sớm** khi đủ 26.

In [12]:
def pangram_bool_array(sentence: str) -> bool:
    seen = [False] * 26
    count = 0
    for ch in sentence:
        idx = ord(ch) - ord("a")
        if not seen[idx]:
            seen[idx] = True
            count += 1
            if count == 26:
                return True      # dừng sớm, khỏi đọc phần còn lại
    return False

print(pangram_bool_array(s1), pangram_bool_array(s2))

True False


## 6. TRACE từng bước — mảng đánh dấu

Với `sentence = "abcabd"` (không pangram), bảng trạng thái:

| bước | ch | idx | seen[idx] trước | hành động | count |
|---:|:---:|---:|:---:|---|---:|
| 0 | a | 0 | False | đánh dấu | 1 |
| 1 | b | 1 | False | đánh dấu | 2 |
| 2 | c | 2 | False | đánh dấu | 3 |
| 3 | a | 0 | True | bỏ qua (đã thấy) | 3 |
| 4 | b | 1 | True | bỏ qua | 3 |
| 5 | d | 3 | False | đánh dấu | 4 |

Kết thúc: `count = 4 < 26` ⇒ `False`.

In [13]:
def trace_pangram(sentence: str) -> bool:
    seen = [False] * 26
    count = 0
    for step, ch in enumerate(sentence):
        idx = ord(ch) - ord("a")
        before = seen[idx]
        if not before:
            seen[idx] = True
            count += 1
            action = "đánh dấu"
        else:
            action = "bỏ qua (đã thấy)"
        print(f"bước {step}: ch={ch} idx={idx:>2} trước={before!s:<5} -> {action:<16} count={count}")
        if count == 26:
            print("Đủ 26 chữ -> True (dừng sớm)")
            return True
    print(f"Hết chuỗi, count={count} < 26 -> False")
    return False

trace_pangram("abcabd")

bước 0: ch=a idx= 0 trước=False -> đánh dấu         count=1
bước 1: ch=b idx= 1 trước=False -> đánh dấu         count=2
bước 2: ch=c idx= 2 trước=False -> đánh dấu         count=3
bước 3: ch=a idx= 0 trước=True  -> bỏ qua (đã thấy) count=3
bước 4: ch=b idx= 1 trước=True  -> bỏ qua (đã thấy) count=3
bước 5: ch=d idx= 3 trước=False -> đánh dấu         count=4
Hết chuỗi, count=4 < 26 -> False


False

Trace thứ hai: chuỗi bảng chữ cái đầy đủ rồi có thêm đuôi `"zzz"` — thấy rõ vòng lặp **dừng sớm** ở bước 25, không đọc đuôi.

In [14]:
r = trace_pangram(string.ascii_lowercase[:22] + "wxyz" + "zzz")
assert r is True

bước 0: ch=a idx= 0 trước=False -> đánh dấu         count=1
bước 1: ch=b idx= 1 trước=False -> đánh dấu         count=2
bước 2: ch=c idx= 2 trước=False -> đánh dấu         count=3
bước 3: ch=d idx= 3 trước=False -> đánh dấu         count=4
bước 4: ch=e idx= 4 trước=False -> đánh dấu         count=5
bước 5: ch=f idx= 5 trước=False -> đánh dấu         count=6
bước 6: ch=g idx= 6 trước=False -> đánh dấu         count=7
bước 7: ch=h idx= 7 trước=False -> đánh dấu         count=8
bước 8: ch=i idx= 8 trước=False -> đánh dấu         count=9
bước 9: ch=j idx= 9 trước=False -> đánh dấu         count=10
bước 10: ch=k idx=10 trước=False -> đánh dấu         count=11
bước 11: ch=l idx=11 trước=False -> đánh dấu         count=12
bước 12: ch=m idx=12 trước=False -> đánh dấu         count=13
bước 13: ch=n idx=13 trước=False -> đánh dấu         count=14
bước 14: ch=o idx=14 trước=False -> đánh dấu         count=15
bước 15: ch=p idx=15 trước=False -> đánh dấu         count=16
bước 16: ch=q idx=16 trước=

## 7. Cách 4 — bitmask (mặt nạ bit)

Một số nguyên có thể coi là **dãy bit**. Dùng 26 bit thấp nhất làm 26 ô đánh dấu: bit thứ `i` bằng 1 nghĩa là đã thấy chữ thứ `i`.

Các toán tử cần biết:

- `1 << i`: số chỉ có bit thứ `i` bật (ví dụ `1 << 3 = 0b1000 = 8`).
- `a | b`: OR từng bit — bật thêm bit.
- `a & b`: AND từng bit — kiểm tra bit.
- `(1 << 26) - 1`: 26 bit thấp đều là 1 (`0b111...1`).
- `bin(x)`: in dạng nhị phân.

In [15]:
print("1 << 3 =", 1 << 3, bin(1 << 3))
mask = 0
mask |= 1 << 0     # thấy 'a'
mask |= 1 << 2     # thấy 'c'
print("mask =", mask, bin(mask))
print("đã thấy 'c'?", (mask & (1 << 2)) != 0, "| đã thấy 'b'?", (mask & (1 << 1)) != 0)
FULL = (1 << 26) - 1
print("FULL =", FULL, "=", bin(FULL), "| số bit 1:", bin(FULL).count("1"))

1 << 3 = 8 0b1000
mask = 5 0b101
đã thấy 'c'? True | đã thấy 'b'? False
FULL = 67108863 = 0b11111111111111111111111111 | số bit 1: 26


In [16]:
def pangram_bitmask(sentence: str) -> bool:
    FULL = (1 << 26) - 1
    mask = 0
    for ch in sentence:
        mask |= 1 << (ord(ch) - ord("a"))
        if mask == FULL:
            return True
    return False

print(pangram_bitmask(s1), pangram_bitmask(s2))

True False


In [17]:
def trace_bitmask(sentence: str):
    mask = 0
    for ch in sentence:
        bit = ord(ch) - ord("a")
        mask |= 1 << bit
        print(f"{ch}: bit {bit:>2} -> mask = {mask:026b}")
    print("số chữ đã thấy =", bin(mask).count("1"))

trace_bitmask("leetcode")

l: bit 11 -> mask = 00000000000000100000000000
e: bit  4 -> mask = 00000000000000100000010000
e: bit  4 -> mask = 00000000000000100000010000
t: bit 19 -> mask = 00000010000000100000010000
c: bit  2 -> mask = 00000010000000100000010100
o: bit 14 -> mask = 00000010000100100000010100
d: bit  3 -> mask = 00000010000100100000011100
e: bit  4 -> mask = 00000010000100100000011100
số chữ đã thấy = 6


`f"{mask:026b}"` định dạng số theo nhị phân (`b`), độ rộng 26, đệm số 0 bên trái. Bit `a` nằm **ngoài cùng bên phải**.

### 7.1. Cách 5 — `set` có dừng sớm và kiểm tra độ dài

Kết hợp hai mẹo: (1) nếu `len(sentence) < 26` trả `False` ngay; (2) thêm dần vào `set` và dừng khi đủ 26.

In [18]:
def pangram_early_exit(sentence: str) -> bool:
    if len(sentence) < 26:
        return False
    seen = set()
    for ch in sentence:
        seen.add(ch)
        if len(seen) == 26:
            return True
    return False

print(pangram_early_exit(s1), pangram_early_exit(s2), pangram_early_exit("a" * 30))

True False False


## 8. Code nộp LeetCode

### Cách A — `set` (ngắn, dễ đọc, khuyên dùng)

In [19]:
class Solution:
    def checkIfPangram(self, sentence: str) -> bool:
        return len(set(sentence)) == 26

print(Solution().checkIfPangram(s1), Solution().checkIfPangram(s2))

True False


### Cách B — mảng đánh dấu (không cần cấu trúc băm)

In [20]:
class SolutionArray:
    def checkIfPangram(self, sentence: str) -> bool:
        seen = [False] * 26
        for ch in sentence:
            seen[ord(ch) - ord("a")] = True
        return all(seen)

print(SolutionArray().checkIfPangram(s1), SolutionArray().checkIfPangram(s2))

True False


### Cách C — bitmask

In [21]:
class SolutionBitmask:
    def checkIfPangram(self, sentence: str) -> bool:
        mask = 0
        for ch in sentence:
            mask |= 1 << (ord(ch) - 97)
        return mask == (1 << 26) - 1

print(SolutionBitmask().checkIfPangram(s1), SolutionBitmask().checkIfPangram(s2))

True False


## 9. Vì sao đúng?

**Invariant** (cho cách set / mảng / bitmask): sau khi đọc `k` ký tự đầu, cấu trúc đánh dấu chứa **đúng** tập chữ cái xuất hiện trong `sentence[:k]` — không thiếu (mỗi ký tự đọc được đều được ghi), không thừa (chỉ ghi ký tự đã đọc).

Khi đọc hết câu, cấu trúc = tập chữ cái của câu. Câu là pangram ⇔ tập này = {a…z} ⇔ (vì chỉ có chữ thường) kích thước = 26 ⇔ `mask == FULL`.

**Dừng sớm đúng** vì: một khi đã đủ 26 thì đọc thêm không thể làm mất chữ nào — kết quả chắc chắn `True`.

**Kiểm tra `len < 26` đúng** vì một chuỗi dài `n` có tối đa `n` ký tự khác nhau.

## 10. Độ phức tạp

Gọi `n = len(sentence)`, `Σ = 26` (kích thước bảng chữ).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. 26 lần `in` | `O(Σ·n)` = `O(n)` | `O(1)` | quét câu nhiều lần |
| 2. `set` | `O(n)` trung bình | `O(Σ)` | ngắn nhất |
| 3. Mảng 26 ô | `O(n)` | `O(Σ)` = `O(1)` | có thể dừng sớm |
| 4. Bitmask | `O(n)` | `O(1)` (một số nguyên) | gọn bộ nhớ nhất |
| 5. Set + dừng sớm | `O(n)` | `O(Σ)` | nhanh khi câu dài |

Vì `Σ` cố định, bộ nhớ của mọi cách đều là hằng số theo `n`.

## 11. Bộ test

Nhóm test: mẫu đề; đúng 26 chữ; thiếu đúng 1 chữ (`z` hoặc `a`); dài nhưng lặp một chữ; độ dài < 26; ngẫu nhiên đối chiếu 5 cách.

In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS ({actual})")

check("mẫu 1", Solution().checkIfPangram(s1), True)
check("mẫu 2", Solution().checkIfPangram(s2), False)
check("đúng 26 chữ", Solution().checkIfPangram(string.ascii_lowercase), True)
check("thiếu z", Solution().checkIfPangram(string.ascii_lowercase[:-1] * 3), False)
check("thiếu a", Solution().checkIfPangram(string.ascii_lowercase[1:] * 3), False)
check("1 ký tự", Solution().checkIfPangram("a"), False)
check("1000 chữ a", Solution().checkIfPangram("a" * 1000), False)
check("bảng chữ đảo ngược", Solution().checkIfPangram(string.ascii_lowercase[::-1]), True)

mẫu 1                    PASS (True)
mẫu 2                    PASS (False)
đúng 26 chữ              PASS (True)
thiếu z                  PASS (False)
thiếu a                  PASS (False)
1 ký tự                  PASS (False)
1000 chữ a               PASS (False)
bảng chữ đảo ngược       PASS (True)


In [23]:
import random
rng = random.Random(1832)
methods = [pangram_check_each_letter, pangram_all, pangram_set, pangram_bool_array,
           pangram_bitmask, pangram_early_exit,
           Solution().checkIfPangram, SolutionArray().checkIfPangram, SolutionBitmask().checkIfPangram]
for _ in range(1000):
    pool = rng.sample(string.ascii_lowercase, rng.randint(20, 26))
    s = "".join(rng.choice(pool) for _ in range(rng.randint(1, 120)))
    expected = set(s) == set(string.ascii_lowercase)
    results = [f(s) for f in methods]
    assert all(r == expected for r in results), (s, results)
print("Đối chiếu 9 cài đặt trên 1000 chuỗi ngẫu nhiên PASS")

Đối chiếu 9 cài đặt trên 1000 chuỗi ngẫu nhiên PASS


`rng.sample(pool, k)` chọn `k` phần tử **không lặp** từ `pool`; `rng.choice(pool)` chọn 1 phần tử. Dùng `random.Random(seed)` để kết quả test lặp lại được.

## 12. Lỗi người mới hay gặp

### Lỗi 1: so `len(sentence) == 26` thay vì số chữ **khác nhau**

In [24]:
def wrong_len_sentence(sentence):
    return len(sentence) >= 26

print("sai :", wrong_len_sentence("a" * 30), "| đúng:", pangram_set("a" * 30))

sai : True | đúng: False


Sửa: đếm ký tự khác nhau bằng `set` (hoặc mảng đánh dấu).

### Lỗi 2: gõ tay bảng chữ cái và gõ sót

In [25]:
def wrong_typed_alphabet(sentence):
    alphabet = "abcdefghijklmnopqrstuvwxy"      # thiếu z!
    return all(ch in sentence for ch in alphabet)

print("sai :", wrong_typed_alphabet(string.ascii_lowercase[:-1]), "| đúng:", pangram_set(string.ascii_lowercase[:-1]))

sai : True | đúng: False


Sửa: dùng `string.ascii_lowercase`.

### Lỗi 3: `return` trong vòng lặp quá sớm

In [26]:
def wrong_early_return(sentence):
    for ch in string.ascii_lowercase:
        if ch in sentence:
            return True          # mới thấy 1 chữ đã kết luận!
        return False

print("sai :", wrong_early_return("abc"), "| đúng:", pangram_set("abc"))

sai : True | đúng: False


Sửa: chỉ `return False` khi **thiếu**; `return True` đặt **sau** vòng lặp.

### Lỗi 4: nhầm `1 << i` với `i << 1`, hoặc dùng `+` thay `|`

In [27]:
def wrong_plus_mask(sentence):
    mask = 0
    for ch in sentence:
        mask += 1 << (ord(ch) - 97)     # cộng lặp lại khi chữ trùng -> tràn sang bit khác
    return mask == (1 << 26) - 1

tricky = "aaa" + string.ascii_lowercase[2:]   # thiếu b nhưng có 3 chữ a: 1+1+1 = 3 = bit a + bit b!
print("sai :", wrong_plus_mask(tricky), "| đúng:", pangram_bitmask(tricky))

sai : True | đúng: False


`(1<<0) + (1<<0) + (1<<0) = 3 = (1<<0) + (1<<1)` — ba chữ `a` "giả làm" cả chữ `a` lẫn chữ `b`. Sửa: dùng `|` (OR) vì bật một bit đã bật thì không đổi.

## 13. Mẹo nhận dạng pattern

- Câu hỏi "có đủ / có thiếu / có trùng **phần tử nào** không" ⇒ pattern **đánh dấu đã thấy**.
- Bảng giá trị **nhỏ và cố định** (26 chữ, 10 chữ số) ⇒ list kích thước cố định hoặc bitmask, chỉ số `ord(ch) - ord('a')`.
- Bảng giá trị lớn/tùy ý ⇒ `set` / `dict`.
- Câu trả lời `True` có thể kết luận sớm ⇒ thêm **dừng sớm** (`return` ngay khi đủ).
- Ràng buộc đếm (cần ≥ 26 ký tự) ⇒ kiểm tra độ dài trước để loại nhanh.

## 14. Biến thể

1. **Trả về các chữ còn thiếu** (sắp xếp a→z).
2. **Câu có chữ hoa, dấu cách, dấu câu** ⇒ `lower()` + chỉ giữ `isalpha()`.
3. **Đoạn con ngắn nhất là pangram** ⇒ cửa sổ trượt (sliding window) — bài khó hơn.
4. **Đếm số chữ khác nhau** ⇒ `len(set(s))`.

In [28]:
def missing_letters(sentence):
    present = set(sentence)
    return [ch for ch in string.ascii_lowercase if ch not in present]

def is_pangram_messy(text):
    letters = {ch for ch in text.lower() if ch.isalpha()}
    return len(letters) == 26

print(missing_letters("leetcode")[:8], "...")
print(is_pangram_messy("The quick brown fox jumps over the lazy dog!"))

['a', 'b', 'f', 'g', 'h', 'i', 'j', 'k'] ...
True


`[... for ... if ...]` là **list comprehension**; `{... for ...}` là **set comprehension**. `ch.isalpha()` trả `True` nếu là chữ cái.

## 15. Bài tập (tự làm trước)

**Bài 1 — dự đoán output:** `len(set("banana"))` bằng bao nhiêu? `bin((1 << 3) | (1 << 0))` in ra gì?

**Bài 2 — tính tay:** với `sentence = "abcdefghijklmnopqrstuvwxyyy"`, cách mảng đánh dấu dừng ở bước nào hay đọc hết? Kết quả?

**Bài 3 — điền TODO:** cách mảng đánh dấu.

In [29]:
def exercise_pangram_array(sentence):
    # TODO 1: seen = [False] * 26
    # TODO 2: với mỗi ch, đánh dấu seen[ord(ch) - ord('a')]
    # TODO 3: trả về all(seen)
    pass

# assert exercise_pangram_array("thequickbrownfoxjumpsoverthelazydog") is True
# assert exercise_pangram_array("leetcode") is False

**Bài 4 — biến thể:** trả về số chữ cái **còn thiếu** để câu thành pangram.

In [30]:
def exercise_missing_count(sentence):
    # TODO: 26 - số chữ khác nhau
    pass

# assert exercise_missing_count("leetcode") == 20
# assert exercise_missing_count("thequickbrownfoxjumpsoverthelazydog") == 0

**Bài 5 — tự viết bitmask:** viết `exercise_pangram_bitmask(sentence)` không dùng `set`, không dùng list.

In [31]:
def exercise_pangram_bitmask(sentence):
    # TODO: mask |= 1 << (ord(ch) - 97); so với (1 << 26) - 1
    pass

# assert exercise_pangram_bitmask("abcdefghijklmnopqrstuvwxyz") is True
# assert exercise_pangram_bitmask("abc") is False

**Bài 6 — mini challenge:** tìm **độ dài đoạn con ngắn nhất** của `s` là pangram (trả `-1` nếu không có). Ví dụ `"aabcdefghijklmnopqrstuvwxyzz"` → `26`.

In [32]:
def exercise_shortest_pangram_window(s):
    # TODO: cửa sổ trượt với dict đếm, co trái khi đủ 26 chữ
    pass

# assert exercise_shortest_pangram_window("aabcdefghijklmnopqrstuvwxyzz") == 26
# assert exercise_shortest_pangram_window("leetcode") == -1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | set bỏ trùng | b, a, n | bit 3 và bit 0 bật |
| 2 | đếm chữ khác nhau | có tới `y` chưa có `z` | đọc hết, `False` |
| 3 | mỗi chữ một ô | `idx = ord(ch) - ord("a")` | `return all(seen)` |
| 4 | đếm khác nhau | `len(set(...))` | `26 - len(set(sentence))` |
| 5 | số nguyên làm 26 ô | toán tử `|=` và `<<` | `return mask == (1 << 26) - 1` |
| 6 | hai con trỏ `left`, `right` | `dict` đếm + `formed` = số chữ có mặt | khi `formed == 26` thì cập nhật đáp án rồi tăng `left` |

## 17. Đáp án tham khảo

**Bài 1:** `len(set("banana")) = 3`; `bin(9) = '0b1001'`.

**Bài 2:** chuỗi có a…y nhưng không có `z` ⇒ không bao giờ đủ 26, đọc hết chuỗi, trả `False`.

In [33]:
print(len(set("banana")), bin((1 << 3) | (1 << 0)))
print(pangram_bool_array("abcdefghijklmnopqrstuvwxyyy"))

3 0b1001
False


In [34]:
def answer_pangram_array(sentence):
    seen = [False] * 26
    for ch in sentence:
        seen[ord(ch) - ord("a")] = True
    return all(seen)


def answer_missing_count(sentence):
    return 26 - len(set(sentence))


def answer_pangram_bitmask(sentence):
    mask = 0
    for ch in sentence:
        mask |= 1 << (ord(ch) - 97)
    return mask == (1 << 26) - 1


def answer_shortest_pangram_window(s):
    count = {}
    formed = 0
    best = float("inf")
    left = 0
    for right, ch in enumerate(s):
        count[ch] = count.get(ch, 0) + 1
        if count[ch] == 1:
            formed += 1
        while formed == 26:
            best = min(best, right - left + 1)
            lc = s[left]
            count[lc] -= 1
            if count[lc] == 0:
                formed -= 1
            left += 1
    return best if best != float("inf") else -1


assert answer_pangram_array("thequickbrownfoxjumpsoverthelazydog") is True
assert answer_pangram_array("leetcode") is False
assert answer_missing_count("leetcode") == 20
assert answer_missing_count("thequickbrownfoxjumpsoverthelazydog") == 0
assert answer_pangram_bitmask("abcdefghijklmnopqrstuvwxyz") is True
assert answer_pangram_bitmask("abc") is False
assert answer_shortest_pangram_window("aabcdefghijklmnopqrstuvwxyzz") == 26
assert answer_shortest_pangram_window("leetcode") == -1
assert answer_shortest_pangram_window("zz" + string.ascii_lowercase[::-1] + "a") == 26
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


`count.get(ch, 0)` trả `count[ch]` nếu có, ngược lại trả `0` — tránh lỗi `KeyError` khi gặp ký tự lần đầu.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def checkIfPangram(self, sentence: str) -> bool:
        return len(set(sentence)) == 26
```

Khung tổng quát "đánh dấu đã thấy":

```python
seen = [False] * 26           # hoặc set(), hoặc mask = 0
for ch in s:
    seen[ord(ch) - ord('a')] = True
return all(seen)
```

Checklist:

- [ ] Đếm chữ **khác nhau**, không phải độ dài chuỗi.
- [ ] Không gõ tay bảng chữ cái — dùng `string.ascii_lowercase` nếu cần.
- [ ] `return False` chỉ khi **thiếu**; `return True` sau vòng lặp (hoặc khi đã đủ 26).
- [ ] Bitmask dùng `|`, không dùng `+`.
- [ ] Nếu input có ký tự lạ (đề khác), dùng phép tập con hoặc lọc `isalpha()`.